In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import os
import sys

import cv2
import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks import LearningRateMonitor

from box import Box
import yaml

import matplotlib.pyplot as plt

from pathlib import Path
import random 
import time 

root_dir = os.path.abspath('..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.dpso_train import DPSO_train as DPSO
from src.data_loader.lightning_module import DPSO_LightningModule
from src.data_loader.data_module_lightning import SonarSimDataModule
from src.data_loader.transforms import SonarDatasetTranforms
from src.data_loader.metrics import pose_err


In [30]:

BASE_DIR = root_dir
# model configuration files
model_config_pth = os.path.join(BASE_DIR, 'config', 'model.yaml')
sonar_config_pth = os.path.join(BASE_DIR, 'config', 'sonar.yaml')

# dataset path
data_dir = os.path.join(BASE_DIR, 'data')

# output data 
output_dir = os.path.join(BASE_DIR, 'traning/output/run_test') 
os.makedirs(output_dir, exist_ok=True)

# === DATASET CONFIGURATION ===
with open(model_config_pth, "r") as f:
            model_config = Box(yaml.safe_load(f))

fls_img_size = (model_config.FLS_INPUT_HEIGHT, model_config.FLS_INPUT_WIDTH)
transforms = SonarDatasetTranforms

# === TRAINING CONFIGURATION ===

# dataloader: 
epochs = 1
batch_size = 1
num_workers = 2

frames_in_series = 10
init_frames = 5 # init_frames < frames_in_series

# model and loss
mode = 'supervised'

# mode = 'unsupervised'

supervised_traning_param = {
    'freeze_poses_steps':1,
    'init_pose_max_noise':2.0,
    'loss_weight_trans':1.0,
    'loss_weight_rot':1.0,
    'loss_weight_proj_r':1.0,
    'loss_weight_proj_theta':1.0
}

selfsupervised_traning_param = {
    'init_pose_max_noise':2.0,
    'loss_weight_proj_r':1.0,
    'loss_weight_proj_theta':1.0
}

if mode == 'supervised':
    traning_param = supervised_traning_param
else: 
    traning_param = selfsupervised_traning_param

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
# init model
dpso = DPSO(model_config_pth, sonar_config_pth, batch_size, frames_in_series, init_frames)
# init lighning wrapper
model = DPSO_LightningModule(dpso, mode, traning_param)
# init data module 
dm = SonarSimDataModule(data_dir, batch_size, num_workers, transforms, frames_in_series)

c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.


In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=output_dir,          
    filename='dpso-{epoch:02d}-{val_loss:.4f}', 
    monitor='val_loss',              
    mode='min',                      
    save_top_k=5,                    
    save_last=True                   
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

trainer = pl.Trainer(
    callbacks = [checkpoint_callback, lr_monitor],
    gradient_clip_val=1.0,
    max_epochs=epochs,             
    accelerator="auto",          
    devices="auto",            
    log_every_n_steps=10,
    # precision="16-mixed",
    profiler="simple"
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


: 

In [ ]:
trainer.fit(model, datamodule=dm)


  | Name  | Type       | Params | Mode  | FLOPs
-----------------------------------------------------
0 | model | DPSO_train | 3.3 M  | train | 0    
-----------------------------------------------------
3.3 M     Trainable params
0         Non-trainable params
3.3 M     Total params
13.283    Total estimated model params size (MB)
112       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]                 

c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\janis\miniconda3\envs\so\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]